In [1]:
import torch
import os
os.chdir('../')

In [2]:
from scipy import linalg
import numpy as np
import os, torch
from tqdm import tqdm
from reports.util import load_config


@torch.no_grad()
def _trace_sqrtm_product(C1: torch.Tensor, C2: torch.Tensor) -> torch.Tensor:
    # Tr sqrtm(C1 @ C2) = Tr sqrt( C1^{1/2} C2 C1^{1/2} )
    s, U = torch.linalg.eigh(C1)                 # C1 = U diag(s) U^T
    s = s.clamp_min(0)
    C1h = (U * s.sqrt()) @ U.t()                 # C1^{1/2}
    M   = C1h @ C2 @ C1h
    w   = torch.linalg.eigvalsh((M + M.t()) * 0.5).clamp_min(0)
    return w.sqrt().sum()

@torch.no_grad()
def calc_fid_stats(mu1, sigma1, mu2, sigma2, eps: float = 1e-6) -> float:
    # 모두 float64 + 동일 device로 정렬
    C1 = torch.as_tensor(sigma1, dtype=torch.float64)
    device = C1.device
    C2 = torch.as_tensor(sigma2, dtype=torch.float64).to(device)
    m1 = torch.as_tensor(mu1,    dtype=torch.float64).to(device).flatten()
    m2 = torch.as_tensor(mu2,    dtype=torch.float64).to(device).flatten()

    D = m1.numel()
    I = torch.eye(D, dtype=torch.float64, device=device)

    # 대칭화 + 정칙화
    C1 = (C1 + C1.t()) * 0.5 + eps * I
    C2 = (C2 + C2.t()) * 0.5 + eps * I

    diff = m1 - m2
    tr_covmean = _trace_sqrtm_product(C1, C2)
    fid = diff.dot(diff) + torch.trace(C1) + torch.trace(C2) - 2.0 * tr_covmean
    return float(fid)

@torch.no_grad()
def calc_fid_pt_dir(pt_dir: str, mu, sigma, eps: float = 1e-6, num=100000, key="inception_feature") -> float:
    # pt_dir에서 'inception_feature'를 모아서 mu1, sigma1 추정 후 FID 계산
    X = []
    for f in tqdm(os.listdir(pt_dir)[:num]):
        if f.endswith(".pt"):
            v = torch.load(os.path.join(pt_dir, f), map_location="cpu").get(key)
            if v is not None:
                X.append(torch.as_tensor(v, dtype=torch.float64).flatten())
    if len(X) < 2:
        raise ValueError("need >=2 features")

    X   = torch.stack(X, 0)                 # [N, D]
    mu1 = X.mean(0)
    Xc  = X - mu1
    sigma1 = (Xc.t() @ Xc) / (X.shape[0] - 1)  # 불편추정

    return calc_fid_stats(mu1, sigma1, mu, sigma, eps=eps)
    #return calculate_frechet_distance(mu1, sigma1, mu, sigma, eps=eps)

def calculate_frechet_distance(mu1, sigma1, mu2, sigma2, eps=1e-6):
    """Numpy implementation of the Frechet Distance.
    The Frechet distance between two multivariate Gaussians X_1 ~ N(mu_1, C_1)
    and X_2 ~ N(mu_2, C_2) is
            d^2 = ||mu_1 - mu_2||^2 + Tr(C_1 + C_2 - 2*sqrt(C_1*C_2)).

    Stable version by Dougal J. Sutherland.

    Params:
    -- mu1   : Numpy array containing the activations of a layer of the
               inception net (like returned by the function 'get_predictions')
               for generated samples.
    -- mu2   : The sample mean over activations, precalculated on an
               representative data set.
    -- sigma1: The covariance matrix over activations for generated samples.
    -- sigma2: The covariance matrix over activations, precalculated on an
               representative data set.

    Returns:
    --   : The Frechet Distance.
    """

    mu1 = np.atleast_1d(mu1)
    mu2 = np.atleast_1d(mu2)

    sigma1 = np.atleast_2d(sigma1)
    sigma2 = np.atleast_2d(sigma2)

    assert mu1.shape == mu2.shape, \
        'Training and test mean vectors have different lengths'
    assert sigma1.shape == sigma2.shape, \
        'Training and test covariances have different dimensions'

    diff = mu1 - mu2

    # Product might be almost singular
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)
    if not np.isfinite(covmean).all():
        msg = ('fid calculation produces singular product; '
               'adding %s to diagonal of cov estimates') % eps
        print(msg)
        offset = np.eye(sigma1.shape[0]) * eps
        covmean = linalg.sqrtm((sigma1 + offset).dot(sigma2 + offset))

    # Numerical error might give slight imaginary component
    if np.iscomplexobj(covmean):
        if not np.allclose(np.diagonal(covmean).imag, 0, atol=1e-3):
            m = np.max(np.abs(covmean.imag))
            raise ValueError('Imaginary component {}'.format(m))
        covmean = covmean.real

    tr_covmean = np.trace(covmean)

    return (diff.dot(diff) + np.trace(sigma1)
            + np.trace(sigma2) - 2 * tr_covmean)

from pathlib import Path
import torch

def get_clip_score(d):
    s = n = 0
    for p in Path(d).rglob('*.pt'):
        try:
            s += torch.load(p, map_location='cpu')['clip_score'].float().mean().item()
            n += 1
        except Exception:
            continue
    if n == 0:
        raise ValueError(f'No clip_score found under: {d}')
    return s / n, n  # (average, file_count)


In [ ]:
pt_dirs = [
            #'samplings/SANA/4.5/3/Dual-Solver/30000/which_0/which_0',
            #'samplings/SANA/4.5/6/Dual-Solver/30000/which_1/which_0',
            #'samplings/SANA/4.5/6/Dual-Solver/30000/which_2/which_0',
            #'samplings/SANA/4.5/6/Dual-Solver/30000/which_3/which_0',
            #'samplings/SANA/4.5/6/Dual-Solver/30000/which_4/which_0',
            #'samplings/SANA/4.5/3/Dual-Solver/30000/which_5/which_0',
            #'samplings/SANA/4.5/3/Dual-Solver/30000/which_6/which_0',
            #'samplings/SANA/4.5/3/Dual-Solver/30000/which_7/which_0',
            #'samplings/SANA/4.5/3/Dual-Solver/30000/which_8/which_0',
            #'samplings/SANA/4.5/3/Dual-Solver/30000/which_9/which_0',
            #'samplings/SANA/4.5/3/Dual-Solver/30000/which_10/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_11/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_12/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_13/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_14/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_15/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_16/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_17/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_18/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_19/which_0',
        ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    #config = load_config(pt_dir)
    data = torch.load('mscoco2014_fid/coco2014_val_30k_fid_stats_clean.pt')
    #data = torch.load('mscoco2014_fid/coco2014_val_30k_fid_stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'], key='clean_inception_feature')
    #score, num = get_clip_score(pt_dir)
    print(pt_dir, fid)
    

100%|██████████| 30001/30001 [00:31<00:00, 962.96it/s] 


samplings/SANA/4.5/3/Dual-Solver/30000/which_11/which_0 22.667991953173782


100%|██████████| 30001/30001 [00:35<00:00, 835.21it/s] 


samplings/SANA/4.5/3/Dual-Solver/30000/which_12/which_0 22.568166437523757


100%|██████████| 30001/30001 [00:37<00:00, 793.72it/s] 


...

samplings/SANA/4.5/3/Dual-Solver/30000/which_17/which_0 22.589304848843142


100%|██████████| 30001/30001 [00:43<00:00, 694.66it/s]


samplings/SANA/4.5/3/Dual-Solver/30000/which_18/which_0 22.31577085784278


100%|██████████| 30001/30001 [00:32<00:00, 928.48it/s] 


samplings/SANA/4.5/3/Dual-Solver/30000/which_19/which_0 24.34620288626894


In [5]:
pt_dirs = [
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_0/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_1/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_2/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_3/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_4/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_5/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_6/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_7/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_8/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_9/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_10/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_11/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_12/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_13/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_14/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_15/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_16/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_17/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_18/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_19/which_0',
        ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    #config = load_config(pt_dir)
    data = torch.load('mscoco2014_fid/coco2014_val_30k_fid_stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'], key='inception_feature')
    #score, num = get_clip_score(pt_dir)
    print(pt_dir, fid)
    

100%|██████████| 30001/30001 [00:25<00:00, 1192.75it/s]


samplings/SANA/4.5/3/Dual-Solver/30000/which_0/which_0 23.984280254482883


100%|██████████| 30001/30001 [00:24<00:00, 1202.68it/s]


samplings/SANA/4.5/3/Dual-Solver/30000/which_1/which_0 23.09698916215814


100%|██████████| 30001/30001 [00:24<00:00, 1203.33it/s]


...

samplings/SANA/4.5/3/Dual-Solver/30000/which_17/which_0 22.398247377865005


100%|██████████| 30001/30001 [00:52<00:00, 567.27it/s]


samplings/SANA/4.5/3/Dual-Solver/30000/which_18/which_0 22.326631368396022


100%|██████████| 30001/30001 [00:40<00:00, 742.35it/s]


samplings/SANA/4.5/3/Dual-Solver/30000/which_19/which_0 23.764048514625927


In [6]:
pt_dirs = [
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_0/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_1/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_2/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_3/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_4/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_5/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_6/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_7/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_8/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_9/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_10/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_11/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_12/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_13/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_14/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_15/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_16/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_17/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_18/which_0',
            'samplings/SANA/4.5/3/Dual-Solver/30000/which_19/which_0',
        ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    #config = load_config(pt_dir)
    #data = torch.load('mscoco2014_fid/coco2014_val_30k_fid_stats_clean.pt')
    #data = torch.load('mscoco2014_fid/coco2014_val_30k_fid_stats.pt')
    #fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'], key='clean_inception_feature')
    score, num = get_clip_score(pt_dir)
    print(pt_dir, score)
    

samplings/SANA/4.5/3/Dual-Solver/30000/which_0/which_0 0.31080335289637245
samplings/SANA/4.5/3/Dual-Solver/30000/which_1/which_0 0.31055667258103686
samplings/SANA/4.5/3/Dual-Solver/30000/which_2/which_0 0.3089043282230695
samplings/SANA/4.5/3/Dual-Solver/30000/which_3/which_0 0.31004286022782324
samplings/SANA/4.5/3/Dual-Solver/30000/which_4/which_0 0.31015850418408714
samplings/SANA/4.5/3/Dual-Solver/30000/which_5/which_0 0.3090567044138908
samplings/SANA/4.5/3/Dual-Solver/30000/which_6/which_0 0.3106031806488832
samplings/SANA/4.5/3/Dual-Solver/30000/which_7/which_0 0.3104940368950367
samplings/SANA/4.5/3/Dual-Solver/30000/which_8/which_0 0.31059235890507697
samplings/SANA/4.5/3/Dual-Solver/30000/which_9/which_0 0.3103750826736291
samplings/SANA/4.5/3/Dual-Solver/30000/which_10/which_0 0.31005393486619
samplings/SANA/4.5/3/Dual-Solver/30000/which_11/which_0 0.3097334309955438
samplings/SANA/4.5/3/Dual-Solver/30000/which_12/which_0 0.3090958285590013
samplings/SANA/4.5/3/Dual-Solver

In [13]:
pt_dirs = [
            #'samplings/SANA/4.5/6/Dual-Solver/30000/which_0/which_0',
            #'samplings/SANA/4.5/6/Dual-Solver/30000/which_1/which_0',
            #'samplings/SANA/4.5/6/Dual-Solver/30000/which_2/which_0',
            #'samplings/SANA/4.5/6/Dual-Solver/30000/which_3/which_0',
            # 'samplings/SANA/4.5/6/Dual-Solver/30000/which_4/which_0',
            # 'samplings/SANA/4.5/6/Dual-Solver/30000/which_5/which_0',
            # 'samplings/SANA/4.5/6/Dual-Solver/30000/which_6/which_0',
            # 'samplings/SANA/4.5/6/Dual-Solver/30000/which_7/which_0',
            # 'samplings/SANA/4.5/6/Dual-Solver/30000/which_8/which_0',
            # 'samplings/SANA/4.5/6/Dual-Solver/30000/which_9/which_0',
            # 'samplings/SANA/4.5/6/Dual-Solver/30000/which_10/which_0',
            # 'samplings/SANA/4.5/6/Dual-Solver/30000/which_11/which_0',
            # 'samplings/SANA/4.5/6/Dual-Solver/30000/which_12/which_0',
            #'samplings/SANA/4.5/6/Dual-Solver/30000/which_13/which_0',
            'samplings/SANA/4.5/6/Dual-Solver/30000/which_14/which_0',
            #'samplings/SANA/4.5/6/Dual-Solver/30000/which_15/which_0',
            # 'samplings/SANA/4.5/6/Dual-Solver/30000/which_16/which_0',
            # 'samplings/SANA/4.5/6/Dual-Solver/30000/which_17/which_0',
            # 'samplings/SANA/4.5/6/Dual-Solver/30000/which_18/which_0',
            # 'samplings/SANA/4.5/6/Dual-Solver/30000/which_19/which_0',
        ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    data = torch.load('mscoco2014_fid/coco2014_val_30k_fid_stats_clean.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'], key='clean_inception_feature')
    score, num = get_clip_score(pt_dir)
    print(pt_dir, fid, score)
    

100%|██████████| 30001/30001 [00:12<00:00, 2359.59it/s]


samplings/SANA/4.5/6/Dual-Solver/30000/which_14/which_0 19.561037891624323 0.31539897505640985


In [19]:
pt_dirs = [
            # 'samplings/SANA/4.5/6/Dual-Solver/30000/which_0/which_0',
            # 'samplings/SANA/4.5/6/Dual-Solver/30000/which_1/which_0',
            # 'samplings/SANA/4.5/6/Dual-Solver/30000/which_2/which_0',
            # 'samplings/SANA/4.5/6/Dual-Solver/30000/which_3/which_0',
            #  'samplings/SANA/4.5/6/Dual-Solver/30000/which_4/which_0',
            #  'samplings/SANA/4.5/6/Dual-Solver/30000/which_5/which_0',
            #  'samplings/SANA/4.5/6/Dual-Solver/30000/which_6/which_0',
            #  'samplings/SANA/4.5/6/Dual-Solver/30000/which_7/which_0',
            #  'samplings/SANA/4.5/6/Dual-Solver/30000/which_8/which_0',
            #  'samplings/SANA/4.5/6/Dual-Solver/30000/which_9/which_0',
            #  'samplings/SANA/4.5/6/Dual-Solver/30000/which_10/which_0',
            #  'samplings/SANA/4.5/6/Dual-Solver/30000/which_11/which_0',
            #  'samplings/SANA/4.5/6/Dual-Solver/30000/which_12/which_0',
            # 'samplings/SANA/4.5/6/Dual-Solver/30000/which_13/which_0',
            # 'samplings/SANA/4.5/6/Dual-Solver/30000/which_14/which_0',
            'samplings/SANA/4.5/6/Dual-Solver/30000/which_15/which_0',
            # 'samplings/SANA/4.5/6/Dual-Solver/30000/which_16/which_0',
            # 'samplings/SANA/4.5/6/Dual-Solver/30000/which_17/which_0',
            # 'samplings/SANA/4.5/6/Dual-Solver/30000/which_18/which_0',
            # 'samplings/SANA/4.5/6/Dual-Solver/30000/which_19/which_0',
        ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    data = torch.load('mscoco2014_fid/coco2014_val_30k_fid_stats.pt')
    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'], key='inception_feature')
    score, num = get_clip_score(pt_dir)
    print(pt_dir, fid, score)
    

  0%|          | 0/30001 [00:00<?, ?it/s]

100%|██████████| 30001/30001 [00:13<00:00, 2289.47it/s]


samplings/SANA/4.5/6/Dual-Solver/30000/which_15/which_0 23.54061597088645 0.31481846162875493
